# Kriging on the Branin 2D function (Octave)

This notebook demonstrates basic Gaussian Process (Kriging) regression using **mlibkriging** on the classic 2D Branin test function.

Steps:
1. Setup mlibkriging
2. Define the Branin function and plot it
3. Build a space-filling design and evaluate it
4. Fit a `Kriging` model
5. Predict on a fine grid and plot mean + uncertainty
6. Inspect model parameters

## 0. Setup

Build the C++ core and the Octave binding from source (skip if already built).
Requires: `cmake`, a C++ compiler, Octave ≥ 6.0.

In [1]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

mlibkriging loaded


## 1. Branin function

The Branin function is a standard benchmark for surrogate modelling, defined on $[0,1]^2$
(rescaled from its canonical domain $[-5, 10] \times [0, 15]$).
It has three global minima.

In [2]:
function z = branin(X)
    x1 = X(:,1) * 15 - 5;
    x2 = X(:,2) * 15;
    z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
        + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
end

% 50x50 evaluation grid
grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);

figure;
contourf(G1, G2, z_true, 20);
colorbar;
title('True Branin function');
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp4zob9ydk/plots/tmpy7x6musr does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 2. Design of experiments

We sample $n = 20$ points using a Latin Hypercube Design.

In [3]:
rand('seed', 42);
n = 20; d = 2;

% Simple LHS: stratified uniform sample, independently permuted per dimension
X = zeros(n, d);
for j = 1:d
    perm = randperm(n);
    X(:,j) = (perm' - rand(n,1)) / n;
end
y = branin(X);

figure;
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title(sprintf('%d LHS design points on Branin', n));
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp4zob9ydk/plots/tmp299yxf7z does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 3. Fit a Kriging model

`Kriging()` fits by maximum likelihood (`objective = "LL"`).
`optim = "BFGS10"` runs 10 random restarts to avoid local optima.

In [4]:
k = Kriging(y, X, 'matern5_2', 'constant', false, 'BFGS10', 'LL');
disp(k.summary())

* data: 20x[0.0297777,0.989901],[0.0161273,0.954095] -> 20x[1.5044,187.953]
* trend constant (est.): 138.079
* variance (est.): 15702.1
* covariance:
  * kernel: matern5_2
  * range (est.): 0.511104, 1.07321
  * fit:
    * objective: LL
    * optim: BFGS10



## 4. Predict on a fine grid

`predict()` returns the posterior mean and standard deviation at new points.

In [5]:
[p_mean, p_stdev] = k.predict(grid_pts, true, false, false);
z_mean = reshape(p_mean, 50, 50);
z_sd   = reshape(p_stdev, 50, 50);

vmin = min(min(z_true(:)), min(z_mean(:)));
vmax = max(max(z_true(:)), max(z_mean(:)));

figure;
subplot(1, 2, 1);
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('True Branin'); xlabel('x_1'); ylabel('x_2');

subplot(1, 2, 2);
contourf(G1, G2, z_mean, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('Kriging mean'); xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp4zob9ydk/plots/tmp1hlti6to does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



In [6]:
% Posterior standard deviation (uncertainty)
figure;
contourf(G1, G2, z_sd, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title('Kriging std dev (uncertainty)');
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp4zob9ydk/plots/tmphizol0_r does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 5. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$, and the log-likelihood at the optimum.

In [7]:
fprintf('Kernel       : %s\n', k.kernel());
fprintf('Theta (range): %s\n', mat2str(k.theta(), 4));
fprintf('Sigma2       : %.4f\n', k.sigma2());
fprintf('LogLikelihood: %.4f\n', k.logLikelihood());

Kernel       : matern5_2


Theta (range): [0.5111;1.073]


Sigma2       : 15702.0672


LogLikelihood: -87.5807


In [8]:
% Log-likelihood surface over (theta1, theta2)
theta_vals = linspace(0.05, 3.0, 40);
[T1, T2] = meshgrid(theta_vals, theta_vals);
ll_mat = zeros(40, 40);
for i = 1:40
    for j = 1:40
        ll_mat(i,j) = k.logLikelihoodFun([T1(i,j); T2(i,j)], false, false);
    end
end

figure;
contourf(T1, T2, ll_mat, 20);
colorbar;
hold on;
th = k.theta();
plot(th(1), th(2), 'r+', 'MarkerSize', 12, 'LineWidth', 2);
hold off;
title('Log-likelihood surface');
xlabel('\theta_1'); ylabel('\theta_2');
legend('', 'optimum');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmp4zob9ydk/plots/tmpoynb3mq0 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13

